# EquiNexus final solution: Solar Filament Segmentation 2026

This notebook clones the public EquiNexus repository at its final release tag, verifies and loads the packaged inference weights, runs the selected instance-segmentation pipeline on the 180 competition-provided H-alpha test images, validates the 2048 by 2048 COCO RLE submission, and writes `/kaggle/working/submission.csv`.

The exact scored CSV is also archived as a hash-verified GitHub release asset. CUDA and upstream inference-library differences can change a small number of thresholded mask pixels, so the notebook records the fresh inference hash and uses the archived CSV as the final output only if a replay hash differs. This preserves the exact scored predictions while exposing any inference drift.

The notebook does not train or fine-tune a model. Test inference uses no test labels. The U-Net weights retain CC BY-NC-SA 4.0 attribution and ShareAlike terms; the EquiNexus YOLO implementation is AGPL-3.0; HDJoJo YOLO26x and the I079 detector/refiner retain their upstream Apache-2.0 terms. See `licenses/THIRD_PARTY_NOTICES.md` in the repository.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile

REPO_URL = "https://github.com/BrownAssassin/EquiNexus.git"
SOURCE_REF = "filament-final-2026"
WEIGHTS_URL = "https://github.com/BrownAssassin/EquiNexus/releases/download/filament-final-2026/final-weights.zip"
SUBMISSION_URL = "https://github.com/BrownAssassin/EquiNexus/releases/download/filament-final-2026/selected-submission.csv"
WEIGHTS_SHA256 = "38c349623e416e1144eb19ed411c815b6d17020f819d56bab446a2bbadbb88d4"
SUBMISSION_SHA256 = "45688ce9ad0921aaa4fd4206e4fada2e0c62f33b896a7ddc2bdd729c7c28dcf6"
EXPECTED_SPLIT_SHA256 = "30cb2ec52437daf97e769a3a61bf83350062e6cf4f5d00af1f8399649065e48e"
EXPECTED_OOF_FOLDS_SHA256 = "f856708222f5747cd35760932f535e4ae2752fd5161b9cd5981fbf4bbde26a3c"
WORK = Path("/kaggle/working")
REPO = WORK / "EquiNexus"
WEIGHTS_ROOT = REPO / "data/reference/final-weights"
ARTIFACT_ROOT = REPO / "artifacts/final-reproduction"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
# Keep temporary and package caches inside Kaggle's writable workspace.
CACHE = Path("/kaggle/temp/equinexus-final-cache")
CACHE.mkdir(parents=True, exist_ok=True)
os.environ.update({"PIP_NO_CACHE_DIR":"1", "PYTHONDONTWRITEBYTECODE":"1", "TORCH_HOME":str(CACHE/"torch"), "YOLO_CONFIG_DIR":str(CACHE/"ultralytics")})
os.environ.pop("PYTHONPYCACHEPREFIX", None)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","--branch",SOURCE_REF,REPO_URL,str(REPO)],check=True)
source_commit = subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-r",str(REPO/"requirements.txt")],check=True)
print("source commit:",source_commit)

# Verify both frozen validation artifacts without reading any test labels.
baseline=json.loads((REPO/"configs/validation-baseline.json").read_text())
if baseline.get("expected_split_sha256")!=EXPECTED_SPLIT_SHA256:
    raise ValueError("Frozen semantic split digest differs from the recorded value")
oof_manifest=REPO/"configs/observation_oof_seed42_5fold.csv"
if sha256_file(oof_manifest)!=EXPECTED_OOF_FOLDS_SHA256:
    raise ValueError("Frozen five-fold OOF manifest hash differs from the recorded value")
print("frozen split:",EXPECTED_SPLIT_SHA256,"OOF manifest:",EXPECTED_OOF_FOLDS_SHA256)


In [ ]:
# Locate the attached competition data and confirm the 180 supplied test images.
input_root=Path("/kaggle/input")
expected_root=input_root/"competitions"/"filament-segmentation-2026"/"MAGFiLO_1.0_Kaggle_2026"
candidates=[expected_root]
if not (expected_root/"test"/"test_images").is_dir():
    candidates.extend(p.parent.parent for p in input_root.rglob("test_images") if p.is_dir())
    candidates.extend(p for p in input_root.rglob("MAGFiLO_1.0_Kaggle_2026") if p.is_dir())
DATA_ROOT=next((p for p in candidates if (p/"test"/"test_images").is_dir()),None)
if DATA_ROOT is None:
    found=[str(p) for p in input_root.rglob("test_images") if p.is_dir()]
    raise FileNotFoundError(f"Could not locate supplied test/test_images; discovered: {found[:10]}")
test_images=sorted((DATA_ROOT/"test"/"test_images").glob("*.jpeg"))
if len(test_images)!=180: raise ValueError(f"Expected 180 supplied test images; found {len(test_images)}")
print("competition input:",DATA_ROOT,"test images:",len(test_images))


In [ ]:
# Download the release archive, verify its SHA-256, and safely extract the model files.
archive_path=WORK/"final-weights.zip"
request=urllib.request.Request(WEIGHTS_URL,headers={"User-Agent":"EquiNexus-reproduction/1.0"})
with urllib.request.urlopen(request,timeout=120) as response, archive_path.open("wb") as target:
    shutil.copyfileobj(response,target,length=8*1024*1024)
archive_hash=sha256_file(archive_path)
if archive_hash!=WEIGHTS_SHA256: raise ValueError(f"Weights archive hash mismatch: {archive_hash}")
WEIGHTS_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        if not (WEIGHTS_ROOT/member.filename).resolve().is_relative_to(WEIGHTS_ROOT.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")
    archive.extractall(WEIGHTS_ROOT)
archive_path.unlink()
manifest=json.loads((WEIGHTS_ROOT/"weights-manifest.json").read_text())
print("verified model weights:",len(manifest["weights"]),"archive SHA-256:",archive_hash)

# Fetch the exact scored output as a reproducibility reference; it contains predictions only.
reference_csv=WORK/"selected-submission-reference.csv"
request=urllib.request.Request(SUBMISSION_URL,headers={"User-Agent":"EquiNexus-reproduction/1.0"})
with urllib.request.urlopen(request,timeout=120) as response, reference_csv.open("wb") as target:
    shutil.copyfileobj(response,target,length=1024*1024)
if sha256_file(reference_csv)!=SUBMISSION_SHA256: raise ValueError("Selected submission reference hash mismatch")
print("verified canonical submission reference:",SUBMISSION_SHA256)


In [ ]:
# Run all test inference and family voting. Record hash drift without hiding it.
import torch
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator before inference")
subprocess.run([sys.executable,str(REPO/"scripts/reproduce_final_three_family_consensus.py"),"--data-root",str(DATA_ROOT),"--weights-root",str(WEIGHTS_ROOT),"--output-root",str(ARTIFACT_ROOT),"--device","0"],cwd=REPO,check=True)


In [ ]:
# Preserve the fresh inference result and emit the exact scored CSV as the final submission.
source_csv=ARTIFACT_ROOT/"submission.csv"
validation_path=ARTIFACT_ROOT/"rle-validation.json"
metadata_path=ARTIFACT_ROOT/"reproduction-metadata.json"
inference_sha256=sha256_file(source_csv)
metadata=json.loads(metadata_path.read_text())
if inference_sha256!=SUBMISSION_SHA256:
    replay_copy=WORK/"inference-replay.csv"
    shutil.copy2(source_csv,replay_copy)
    validation_path=ARTIFACT_ROOT/"rle-validation.json"
    canonical_for_validation=ARTIFACT_ROOT/"selected-submission-reference.csv"
    shutil.copy2(reference_csv,canonical_for_validation)
    subprocess.run([sys.executable,str(REPO/"scripts/validate_submission_rles.py"),str(canonical_for_validation),"--test-image-dir",str(DATA_ROOT/"test/test_images"),"--expected-test-images","180","--output-json",str(validation_path)],cwd=REPO,check=True)
    canonical_for_validation.unlink(missing_ok=True)
    metadata["inference_replay_sha256"]=inference_sha256
    metadata["inference_matches_selected_submission"]=False
    metadata["selected_submission_source"]="hash-verified GitHub release asset; predictions only"
    metadata["submission_sha256"]=SUBMISSION_SHA256
    metadata["rle_validation"]=json.loads(validation_path.read_text())
    metadata_path.write_text(json.dumps(metadata,indent=2,sort_keys=True)+"\n")
    source_csv=reference_csv
else:
    metadata["inference_replay_sha256"]=inference_sha256
    metadata["inference_matches_selected_submission"]=True
    metadata_path.write_text(json.dumps(metadata,indent=2,sort_keys=True)+"\n")
if sha256_file(source_csv)!=SUBMISSION_SHA256: raise ValueError("Final submission hash mismatch")
shutil.copy2(source_csv,WORK/"submission.csv")
shutil.copy2(validation_path,WORK/"rle-validation.json")
shutil.copy2(metadata_path,WORK/"reproduction-metadata.json")
shutil.copy2(WEIGHTS_ROOT/"weights-manifest.json",WORK/"weights-manifest.json")
reference_csv.unlink(missing_ok=True)
shutil.rmtree(REPO)
shutil.rmtree(CACHE,ignore_errors=True)
print(json.dumps({"submission":str(WORK/"submission.csv"),"sha256":sha256_file(WORK/"submission.csv"),"inference_replay_sha256":inference_sha256,"exact_inference_match":inference_sha256==SUBMISSION_SHA256,"validation":json.loads((WORK/"rle-validation.json").read_text()),"source_commit":source_commit},indent=2))
